In [0]:
%sql
USE CATALOG mvp_pipeline;

In [0]:
%sql
-- ICMS por CNAE subclasse
CREATE OR REPLACE TABLE silver.icms_cnae_subclasse AS
SELECT
  CAST(ano AS INT)                                        AS ano,
  CAST(mes AS INT)                                        AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)        AS data_ref,
  lpad(regexp_replace(cod_subclasse, '\\D', ''), 7, '0')  AS cnae_subclasse,
  lpad(regexp_replace(cod_subclasse, '\\D', ''), 7, '0') = '0000000' AS sem_cnae,
  trim(nome_subclasse)                                    AS nome_cnae_subclasse,
  cod_versao                                              AS versao_cnae,
  CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_icms,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                     AS _processamento_ts
FROM bronze.icms_cnae_subclasse;

-- Verificação

SELECT sem_cnae, count(*) AS linhas, count(DISTINCT versao_cnae) AS versoes,
       round(sum(valor_icms)/1e9, 2) AS valor_bi
FROM silver.icms_cnae_subclasse GROUP BY sem_cnae;

In [0]:
%sql

-- Arrecadação por município e COREDE

CREATE OR REPLACE TABLE silver.arrecadacao_municipio AS
SELECT
  CAST(ano AS INT)                                 AS ano,
  CAST(mes AS INT)                                 AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1) AS data_ref,
  CAST(cod_corede AS INT)                          AS cod_corede,
  upper(trim(nome_corede))                         AS nome_corede,
  CAST(cod_munic AS INT)                           AS cod_munic_sefaz,
  trim(nome_munic)                                 AS nome_municipio,
  upper(trim(sigla_tipo_arr))                      AS tributo,
  CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_arrecadado,
  _ingestao_ts,
  _fonte,
  current_timestamp()                              AS _processamento_ts
FROM bronze.arrecadacao_municipio_corede;

-- Verificação

SELECT tributo, count(*) AS linhas, min(ano) AS ano_min, max(ano) AS ano_max,
       count(DISTINCT cod_munic_sefaz) AS municipios
FROM silver.arrecadacao_municipio
GROUP BY tributo ORDER BY tributo;

In [0]:
%sql

-- Desonerações (2021 a 2024)

CREATE OR REPLACE TABLE silver.desoneracoes AS
WITH base AS (
  SELECT * FROM bronze.desoneracoes_2021_2023
  UNION ALL
  SELECT * FROM bronze.desoneracoes_2024
)
SELECT
  CAST(ano AS INT)                                    AS ano,
  upper(trim(imposto))                                AS imposto,
  trim(tipo_benef)                                    AS tipo_beneficio,
  CAST(cod_benef AS INT)                              AS cod_beneficio,
  trim(descr_benef)                                   AS descr_beneficio,
  trim(legislacao_aplicada)                           AS legislacao,
  trim(finalidade)                                    AS finalidade,
  trim(justificativa)                                 AS justificativa,
  upper(trim(corede))                                 AS nome_corede,
  trim(cnae_1)                                        AS cnae_secao,
  trim(cnae_2)                                        AS cnae_divisao,
  trim(cnae_3)                                        AS cnae_grupo,
  trim(cnae_5)                                        AS cnae_classe,
  lpad(regexp_replace(cnae_7, '\\D', ''), 7, '0')     AS cnae_subclasse,
  trim(descr_cnae_7)                                  AS nome_cnae_subclasse,
  CAST(replace(replace(vlr_desoner, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_desonerado,
  CAST(qtd_cnpj8 AS INT)                              AS qtd_empresas,
  -- a fonte mistura dois níveis de granularidade na mesma tabela
  CASE WHEN upper(trim(corede)) = 'SEM COREDE' THEN 'AGREGADO_ESTADUAL'
       ELSE 'DETALHADO' END                           AS nivel_agregacao,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                 AS _processamento_ts
FROM base;

-- Verificação

SELECT ano, count(*) AS linhas, sum(valor_desonerado) AS valor_total,
       count(DISTINCT nome_corede) AS coredes,
       count(DISTINCT cnae_subclasse) AS subclasses
FROM silver.desoneracoes
GROUP BY ano ORDER BY ano;

-- Separa o agregado
SELECT nivel_agregacao, count(*) AS linhas,
       sum(valor_desonerado) AS valor,
       round(100 * sum(valor_desonerado) / sum(sum(valor_desonerado)) OVER (), 1) AS perc,
       count(DISTINCT nome_corede) AS coredes,
       count(DISTINCT cnae_subclasse) AS subclasses
FROM silver.desoneracoes
GROUP BY nivel_agregacao;

In [0]:
%sql

-- Cadastro de contribuintes por setor

CREATE OR REPLACE TABLE silver.cadastro_setor AS
SELECT
  CAST(ano AS INT)                                         AS ano,
  CAST(mes AS INT)                                         AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)         AS data_ref,
  trim(categoria)                                          AS categoria,
  lpad(regexp_extract(cnae_fiscal, '^(\\d+)', 1), 7, '0')  AS cnae_subclasse,
  trim(regexp_replace(cnae_fiscal, '^\\d+\\s*-\\s*', ''))  AS nome_cnae_subclasse,
  regexp_extract(cnae_divisao, '^(\\d+)', 1)               AS cod_cnae_divisao,
  trim(regexp_replace(cnae_divisao, '^\\d+\\s*-\\s*', '')) AS nome_cnae_divisao,
  trim(atividade)                                          AS atividade,
  trim(area)                                               AS area,
  trim(setor)                                              AS setor,
  CAST(estabelecimentos_ativos AS INT)                     AS qtd_ativos,
  CAST(estabelecimentos_baixados AS INT)                   AS qtd_baixados,
  CAST(estabelecimentos_novos AS INT)                      AS qtd_novos,
  -- marca a fotografia de fim de ano: qtd_ativos não pode ser somado entre meses
  CAST(mes AS INT) = max(CAST(mes AS INT)) OVER (PARTITION BY CAST(ano AS INT)) AS snapshot_ano,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                      AS _processamento_ts
FROM bronze.cadastro_contribuintes_setor;

-- Verificação

-- 1. categorias do cadastro: define a proxy de MPE
SELECT categoria, count(*) AS linhas, sum(qtd_ativos) AS soma_ativos
FROM silver.cadastro_setor
GROUP BY categoria ORDER BY soma_ativos DESC;

-- 2. CNAEs que não puderam ser extraídos do campo misto
SELECT count(*) AS cnae_invalido
FROM silver.cadastro_setor
WHERE cnae_subclasse = '0000000';

In [0]:
%sql

-- Cadastro de contribuintes por município

CREATE OR REPLACE TABLE silver.cadastro_municipio AS
SELECT
  CAST(ano AS INT)                                             AS ano,
  CAST(mes AS INT)                                             AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)             AS data_ref,
  CAST(cod_categ AS INT)                                       AS cod_categoria,
  trim(categoria)                                              AS categoria,
  lpad(regexp_replace(cod_municipio_ibge, '\\D', ''), 7, '0')  AS cod_municipio_ibge,
  trim(municipio_ibge)                                         AS nome_municipio,
  CAST(cod_corede AS INT)                                      AS cod_corede,
  upper(trim(corede))                                          AS nome_corede,
  CAST(qtd_ativos AS INT)                                      AS qtd_ativos,
  CAST(qtd_baixados AS INT)                                    AS qtd_baixados,
  CAST(qtd_novos AS INT)                                       AS qtd_novos,
  -- mesma regra da célula 5: qtd_ativos é estoque, só a fotografia de fim de ano é comparável
  CAST(mes AS INT) = max(CAST(mes AS INT)) OVER (PARTITION BY CAST(ano AS INT)) AS snapshot_ano,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                          AS _processamento_ts
FROM bronze.cadastro_contribuintes_municipio;

-- Verificação

SELECT count(DISTINCT cod_municipio_ibge) AS municipios,
       count(DISTINCT nome_corede)        AS coredes,
       min(ano) AS ano_min, max(ano) AS ano_max
FROM silver.cadastro_municipio;

In [0]:
%sql

-- De-para CNAE x cadeia produtiva

CREATE OR REPLACE TABLE silver.cnae_cadeia AS
SELECT DISTINCT
  lpad(regexp_replace(cnae_origem, '\\D', ''), 7, '0') AS cnae_subclasse,
  trim(denominacao_cnae)                               AS nome_cnae_subclasse,
  trim(cadeia)                                         AS cadeia,
  trim(cadeia) <> 'Outros (não priorizado)'            AS cadeia_prioritaria,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                  AS _processamento_ts
FROM bronze.cnae_cadeia_produtiva
WHERE cnae_origem IS NOT NULL;

-- Verificação
SELECT cadeia, count(*) AS cnaes
FROM silver.cnae_cadeia GROUP BY cadeia ORDER BY cnaes DESC;

-- CNAEs em mais de uma cadeia
SELECT count(*) AS cnaes_em_duas_cadeias FROM (
  SELECT cnae_subclasse FROM silver.cnae_cadeia
  WHERE cadeia_prioritaria
  GROUP BY cnae_subclasse HAVING count(DISTINCT cadeia) > 1
);

-- Todos os códigos devem ter 7 dígitos
SELECT count(*) AS fora_do_padrao FROM silver.cnae_cadeia
WHERE length(cnae_subclasse) <> 7;

In [0]:
%sql

-- IBGE: municípios e PIB municipal

CREATE OR REPLACE TABLE silver.municipio_ibge AS
SELECT
  lpad(CAST(id AS STRING), 7, '0') AS cod_municipio_ibge,
  trim(nome)                       AS nome_municipio,
  microrregiao.nome                AS microrregiao,
  microrregiao.mesorregiao.nome    AS mesorregiao,
  current_timestamp()              AS _processamento_ts
FROM bronze.ibge_municipios_rs;

-- PIB aninhado: converter em mapa e depois em linhas
CREATE OR REPLACE TABLE silver.pib_municipal AS
SELECT
  lpad(s.localidade.id, 7, '0')             AS cod_municipio_ibge,
  s.localidade.nome                         AS nome_municipio,
  CAST(ano_serie AS INT)                    AS ano,
  CAST(valor_serie AS DECIMAL(18,2))        AS pib_mil_reais,
  CAST(valor_serie AS DECIMAL(18,2)) * 1000 AS pib_reais,
  current_timestamp()                       AS _processamento_ts
FROM bronze.ibge_pib_municipal_rs
LATERAL VIEW explode(resultados) v_res AS res
LATERAL VIEW explode(res.series) v_ser AS s
LATERAL VIEW explode(from_json(to_json(s.serie), 'map<string,string>')) v_pib AS ano_serie, valor_serie;

-- Verificação

SELECT ano, count(*) AS municipios, sum(pib_reais) AS pib_total
FROM silver.pib_municipal GROUP BY ano ORDER BY ano;

In [0]:
%sql

-- De-para de município (SEFAZ x IBGE)

CREATE OR REPLACE TABLE silver.depara_municipio AS
WITH sefaz AS (
  SELECT DISTINCT cod_munic_sefaz,
         nome_municipio,
         regexp_replace(
           translate(upper(nome_municipio),
                     'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ',
                     'AAAAAEEEEIIIIOOOOOUUUUC'),
           '[^A-Z0-9]', '') AS chave
  FROM silver.arrecadacao_municipio
),
ibge AS (
  SELECT DISTINCT cod_municipio_ibge,
         nome_municipio AS nome_ibge,
         regexp_replace(
           translate(upper(nome_municipio),
                     'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ',
                     'AAAAAEEEEIIIIOOOOOUUUUC'),
           '[^A-Z0-9]', '') AS chave
  FROM silver.cadastro_municipio
)
SELECT s.cod_munic_sefaz,
       s.nome_municipio,
       i.cod_municipio_ibge,
       i.nome_ibge,
       -- códigos residuais da SEFAZ que não representam municípios
       s.cod_munic_sefaz IN (0, 900) AS pseudo_municipio,
       current_timestamp() AS _processamento_ts
FROM sefaz s
LEFT JOIN ibge i ON s.chave = i.chave;

-- Verificação

SELECT count(*) AS sem_correspondencia
FROM silver.depara_municipio
WHERE cod_municipio_ibge IS NULL AND NOT pseudo_municipio;

SELECT cod_munic_sefaz, nome_municipio FROM silver.depara_municipio
WHERE cod_municipio_ibge IS NULL AND NOT pseudo_municipio
ORDER BY nome_municipio;

In [0]:
%sql

-- Verificação final da camada

SELECT 'icms_cnae_subclasse' AS tabela, count(*) AS linhas FROM silver.icms_cnae_subclasse
UNION ALL SELECT 'arrecadacao_municipio', count(*) FROM silver.arrecadacao_municipio
UNION ALL SELECT 'desoneracoes',          count(*) FROM silver.desoneracoes
UNION ALL SELECT 'cadastro_setor',        count(*) FROM silver.cadastro_setor
UNION ALL SELECT 'cadastro_municipio',    count(*) FROM silver.cadastro_municipio
UNION ALL SELECT 'cnae_cadeia',           count(*) FROM silver.cnae_cadeia
UNION ALL SELECT 'municipio_ibge',        count(*) FROM silver.municipio_ibge
UNION ALL SELECT 'pib_municipal',         count(*) FROM silver.pib_municipal
UNION ALL SELECT 'depara_municipio',      count(*) FROM silver.depara_municipio
ORDER BY tabela;